In [1]:
import importlib

from sklearn.metrics import mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import time

from scipy.optimize import differential_evolution
import pyswarms as ps

import auxiliar as aux

importlib.reload(aux)

<module 'auxiliar' from 'c:\\Users\\mateu\\OneDrive\\Área de Trabalho\\TimeSeriesEV\\auxiliar.py'>

In [2]:
configs_pso = [
    {
        "n_particles": 50,
        "iters": 200,
        "ftol_iter": 20,
        "c1": 2,
        "c2": 2,
        "w": 0.7,
        "k": 3,
        "p": 2
    }
]

configs_de = [
    {
        "pop_size": 24,
        "iters": 200,
        "mutation": (0.6, 0.9),
        "recombination": 0.7
    }
]

In [3]:
df_GT = pd.read_csv('./dataset/GlobalTemperatures.csv')
df_GT['year'] = df_GT['dt'].str.split('-').str[0].astype(int)
df_GT = df_GT.query("year >= 1800")

df_USR = pd.read_csv('./dataset/USR.csv')
df_USR = df_USR[['Year', 'Month', 'Total Renewable Energy']].groupby(['Year', 'Month']).sum().reset_index()
df_USR['dt'] = df_USR['Year'].astype(str) + '-' + df_USR['Month'].astype(str)

# Global Temperatures

In [4]:
# plt.plot(df_GT['dt'], df_GT['LandAverageTemperature'])

# USR

In [5]:
# plt.plot(df_USR['dt'], df_USR['Total Renewable Energy'])

In [6]:
fitness_GT = aux.create_fitness(df_GT, value_col='LandAverageTemperature')
fitness_USR = aux.create_fitness(df_USR, value_col='Total Renewable Energy')

In [7]:
def PSO_lbest(fitness, n_particles=50, iters=200, ftol_iter=20, c1=2, c2=2, w=0.7, k=3, p=2, seed=42, plot=False):
    # Função objetivo: recebe todas as partículas
    def objective_function(particles):

        costs = []

        for particle in particles:
            cost = fitness(particle)
            costs.append(cost)

        return np.array(costs)


    # Espaço de busca
    lb = np.array([3, 1, 0.001])
    ub = np.array([10, 50, 0.1])

    bounds = (lb, ub)


    # Parâmetros do LBest PSO
    options = {
        'c1': c1,
        'c2': c2,
        'w': w,
        'k': k,  # Número de vizinhos
        'p': p   # Distância euclidiana
    }


    # Reprodutibilidade
    np.random.seed(seed)


    # Inicialização
    optimizer = ps.single.LocalBestPSO(
        n_particles=n_particles,
        dimensions=3,
        options=options,
        bounds=bounds,
        ftol=1e-6,
        ftol_iter=ftol_iter
    )


    # Otimização
    best_cost, best_position = optimizer.optimize(
        objective_function,
        iters=iters
    )

    # Melhores parâmetros
    best_k = int(round(best_position[0]))
    best_n = int(round(best_position[1]))
    best_a = best_position[2]
    
    if plot:

        # Convergência
        plt.figure(figsize=(10, 5))

        plt.plot(optimizer.cost_history)

        plt.xlabel('Iteração')
        plt.ylabel('Melhor MSE global encontrado')
        plt.title('Convergência do LBest PSO')

        plt.grid(alpha=0.3)
        plt.show()

    return best_cost, best_k, best_n, best_a

In [8]:
def DE_DEBEST1(fitness, pop_size=24, iters=200, mutation=(0.6, 0.9), recombination=0.7, seed=42, plot=False):
# Função objetivo
    def objective_function(params):
        return fitness(params)


    # Limites das variáveis
    bounds = [
        (3, 10),   # k: quantidade de lags
        (1, 50),   # n: neurônios escondidos
        (0.001, 0.1)    # a: learning rate init
    ]


    # Histórico de convergência
    history = []

    def callback(intermediate_result):
        history.append(intermediate_result.fun)


    # Executa DE
    result = differential_evolution(
        func=objective_function,
        bounds=bounds,

        strategy='best1bin',

        maxiter=iters,
        popsize=pop_size,

        mutation=mutation,
        recombination=recombination,

        tol=1e-5,
        atol=0,

        integrality=[True, True, False],

        polish=False,
        seed=seed,
        disp=True,
        callback=callback
    )


    # Resultados
    best_k = int(result.x[0])
    best_n = int(result.x[1])
    best_a = result.x[2]

    if plot:
        # Convergência
        plt.figure(figsize=(10, 5))

        plt.plot(history)

        plt.xlabel('Geração')
        plt.ylabel('Melhor MSE')
        plt.title('Convergência do DE/best/1/bin')

        plt.grid(alpha=0.3)
        plt.show()

    return result.fun, best_k, best_n, best_a

In [9]:
def run_experiments(
    algorithm,
    df,
    dataset_name,
    configs,
    n_runs=10,
    base_seed=2026,
    value_col="LandAverageTemperature",
    output_dir="resultados"
):
    """
    Executa múltiplas configurações de PSO ou DE em uma base.

    Cada configuração é executada n_runs vezes, com sementes distintas.
    Salva um CSV com os resultados individuais.
    """

    algorithm = algorithm.lower()

    # Função objetivo da base
    fitness = aux.create_fitness(df, value_col=value_col)

    # Sementes reproduzíveis e distintas
    seeds = [base_seed + i for i in range(n_runs)]

    results = []

    for config_id, config in enumerate(configs, start=1):

        print(
            f"\n{algorithm.upper()} | {dataset_name} | "
            f"Configuração {config_id}/{len(configs)}"
        )

        for run_id, seed in enumerate(seeds, start=1):

            print(f"Execução {run_id}/{n_runs} | seed={seed}")

            start_time = time.perf_counter()

            if algorithm == "pso":
                best_cost, best_k, best_n, best_a = PSO_lbest(
                    fitness,
                    seed=seed,
                    plot=False,
                    **config
                )

            else:
                best_cost, best_k, best_n, best_a = DE_DEBEST1(
                    fitness,
                    seed=seed,
                    plot=False,
                    **config
                )

            elapsed_time = time.perf_counter() - start_time

            results.append({
                "algorithm": algorithm.upper(),
                "dataset": dataset_name,
                "config_id": config_id,
                "run": run_id,
                "seed": seed,
                **config,
                "best_rmse": best_cost,
                "best_k": best_k,
                "best_n": best_n,
                "best_learning_rate": best_a,
                "time_seconds": elapsed_time
            })

    # Organiza os resultados
    results_df = pd.DataFrame(results)

    # Cria diretório, caso não exista
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    # Um CSV por algoritmo e base
    filename = f"{algorithm}_{dataset_name}.csv"
    filepath = output_path / filename

    results_df.to_csv(filepath, index=False, encoding="utf-8-sig")

    print(f"\nResultados salvos em: {filepath}")

    return results_df


In [ ]:
results_pso_GT = run_experiments(
    algorithm="pso",
    df=df_GT,
    dataset_name="temperature",
    configs=configs_pso,
    n_runs=10,
    value_col="LandAverageTemperature"
)

2026-11-16 03:57:24,556 - pyswarms.single.local_best - INFO - Optimize for 200 iters with {'c1': 2, 'c2': 2, 'w': 0.7, 'k': 3, 'p': 2}



PSO | temperature | Configuração 1/1
Execução 1/10 | seed=2026


pyswarms.single.local_best:  12%|█▏        |23/200, best_cost=0.369
2026-11-16 04:02:28,183 - pyswarms.single.local_best - INFO - Optimization finished | best cost: 0.36894627214186787, best pos: [9.97752077 4.28644221 0.05840588]
2026-11-16 04:02:28,190 - pyswarms.single.local_best - INFO - Optimize for 200 iters with {'c1': 2, 'c2': 2, 'w': 0.7, 'k': 3, 'p': 2}


Execução 2/10 | seed=2027


pyswarms.single.local_best:   9%|▉         |18/200, best_cost=0.358c:\Users\mateu\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:792: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
pyswarms.single.local_best:  10%|▉         |19/200, best_cost=0.358

In [ ]:
results_de_GT = run_experiments(
    algorithm="de",
    df=df_GT,
    dataset_name="temperature",
    configs=configs_de,
    n_runs=10,
    value_col="LandAverageTemperature"
)

In [ ]:
results_pso_energy = run_experiments(
    algorithm="pso",
    df=df_USR,
    dataset_name="energy",
    configs=configs_pso,
    n_runs=10,
    value_col="'Total Renewable Energy'"
)

In [ ]:
results_de_energy = run_experiments(
    algorithm="de",
    df=df_USR,
    dataset_name="energy",
    configs=configs_de,
    n_runs=10,
    value_col="'Total Renewable Energy'"
)